In [ ]:
#!pip install transformers datasets accelerate evaluate torch tqdm


In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments, AdamW, get_linear_schedule_with_warmup
from datasets import load_dataset
from tqdm import tqdm
from torch.nn import functional as F
import numpy as np

In [ ]:
model_name = "Salesforce/codet5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [ ]:
### Load and Split Dataset

dataset = load_dataset("csv", data_files="debug_data.csv")["train"]

def group_split(dataset, test_size=0.1, val_size=0.1, seed=42):
    base_ids = list(set(dataset["base_id"]))
    rng = np.random.default_rng(seed)
    shuffled = rng.permutation(base_ids)

    n = len(shuffled)
    n_test = int(test_size * n)
    n_val = int(val_size * n)

    test_groups = set(shuffled[:n_test])
    val_groups  = set(shuffled[n_test:n_test+n_val])
    #train_groups = set(shuffled[n_test+n_val:])

    def label_row(example):
        if example["base_id"] in test_groups:
            return {"split": "test"}
        elif example["base_id"] in val_groups:
            return {"split": "val"}
        else:
            return {"split": "train"}

    dataset = dataset.map(label_row)

    train_ds = dataset.filter(lambda x: x["split"] == "train").remove_columns("split")
    val_ds   = dataset.filter(lambda x: x["split"] == "val").remove_columns("split")
    test_ds  = dataset.filter(lambda x: x["split"] == "test").remove_columns("split")

    return train_ds, val_ds, test_ds

train_ds, val_ds, test_ds = group_split(dataset)
print(f"Datatset Sizes: \nTraining: {len(train_ds)}, Validation: {len(val_ds)}, Test: {len(test_ds)}")

In [ ]:
### Preprocess and Tokenize the datasets

def preprocess_and_tokenize(batch):
    input_texts = [
        f"Fix the following SysML code.\n### Faulty Code:\n{wc}\n### Error:\n{err}"
        for wc, err in zip(batch["wrong_code"], batch["error"])
    ]
    target_texts = batch["correct_code"]

    model_inputs = tokenizer(input_texts, truncation=False)
    labels = tokenizer(target_texts, truncation=False)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = train_ds.map(preprocess_and_tokenize, batched=True)
val_ds   = val_ds.map(preprocess_and_tokenize, batched=True)
test_ds  = test_ds.map(preprocess_and_tokenize, batched=True)

In [ ]:
# Truncate over-length token sequences

def check_and_truncate_tokens(
    dataset,
    id_column: str ="id",
    token_columns: list = ["input_ids", "labels"],
    limit: int = 1024,
):
    
    total_truncated = 0
    truncation_details = []

    for idx, example in enumerate(dataset):
        example_id = example.get(id_column, f"row_{idx}")

        for col in token_columns:
            if col not in example:
                raise ValueError(f"Column '{col}' not found in dataset.")
            
            tokens = example[col]
            if not isinstance(tokens, (list, tuple)):
                raise ValueError(f"Column '{col}' should contain a list of token IDs.")

            token_len = len(tokens)

            if token_len > limit:
                truncated_len = token_len - limit
                example[col] = tokens[:limit] 
                truncation_details.append({
                    "index": idx,
                    "id": example_id,
                    "column": col,
                    "original_length": token_len,
                    "truncated_to": limit,
                    "tokens_removed": truncated_len,
                })
                total_truncated += 1

    if total_truncated > 0:
        print(f"Truncated {total_truncated} over-length sequences to {limit} tokens.")
        print("Examples (up to 5 shown):")
        for entry in truncation_details[:5]:
            print(entry)
    else:
        print(f"No truncations needed. All sequences within {limit} tokens.")

    return dataset, {
        "num_truncated": total_truncated,
        "details": truncation_details
    }

train_dataset, train_report = check_and_truncate_tokens(dataset=train_ds)
val_dataset, val_report = check_and_truncate_tokens(dataset=val_ds)
test_dataset, test_report = check_and_truncate_tokens(dataset=test_ds)

In [ ]:
### Create DataLoaders

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

def create_dataloader(dataset, batch_size=4, shuffle=True):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=data_collator
    )

train_loader = create_dataloader(train_dataset, batch_size=4, shuffle=True)
val_loader = create_dataloader(val_dataset, batch_size=4, shuffle=False)
test_loader = create_dataloader(test_dataset, batch_size=4, shuffle=False)

In [ ]:
batch = next(iter(train_loader)) # sanity check
for k, v in batch.items():
    print(k, v.shape)

#Do test train split
# Incorporate AST based loss
# Compute total loss
# Back propagate
# Test

In [ ]:
def compute_ast_weight(pred_code: str, gold_code: str) -> float: #TODO
    """Compute AST similarity weight between predicted and ground truth code."""
    try:
        ast_pred = generate_ast(pred_code)
        ast_gold = generate_ast(gold_code)
        # Compare ASTs (placeholder — replace with your own)
        # For now: higher weight for larger difference
        similarity = ast_pred.similarity(ast_gold)  # hypothetical method
        weight = 1.0 + (1.0 - similarity)           # weight >1 for larger structural diff
    except Exception:
        # If parsing fails (e.g. syntax error), assign penalty weight
        weight = 2.0
    return weight


In [ ]:
def compute_ast_weights(model, labels, batch):
    model.eval()

    preds = model.generate(batch["input_ids"])
    pred_texts = tokenizer.batch_decode(preds, skip_special_tokens=True)
    correct_texts = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute AST weights per example
    ast_weights = []
    for pred_code, gold_code in zip(pred_texts, correct_texts):
        w = compute_ast_weight(pred_code, gold_code)
        ast_weights.append(w)
    ast_weights = torch.tensor(ast_weights, device=device)

    model.train()

    return ast_weights


def compute_ce_loss(logits, labels):
    # Standard CrossEntropyLoss with padding ignored
    loss_fct = torch.nn.CrossEntropyLoss(ignore_index=-100, reduction="none")

    # Shift for decoder (predict next token)
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()

    # Per-token CE loss
    ce_loss_per_token = loss_fct(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1)
    )

    # Reshape back to [batch, seq_len-1]
    ce_loss_per_token = ce_loss_per_token.view(labels.size(0), -1)

    # Per-example average CE
    ce_loss_per_example = ce_loss_per_token.mean(dim=1)

    return ce_loss_per_example


def compute_loss(model, outputs, batch):

    logits = outputs.logits  # [batch, seq_len, vocab_size]
    labels = batch["labels"]  # [batch, seq_len]

    ce_loss_per_example = compute_ce_loss(logits, labels)
    ast_weights = compute_ast_weights(model, labels, batch)

    weighted_loss = (ce_loss_per_example * ast_weights)

    return weighted_loss.mean()

In [ ]:
num_epochs = 3
optimizer = AdamW(model.parameters(), lr=5e-5)
num_training_steps = len(train_loader) * num_epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

train_losses, val_losses = [], []


def run_epoch(dataloader, evaluating: bool=False):

    running_loss, count = 0.0, 0

    if evaluating:
        model.eval()
        with torch.no_grad():
            for batch in tqdm(dataloader):
                batch = {k: v.to(device) for k, v in batch.items()}
                
                outputs = model(**batch)
                batch_loss = compute_loss(model, outputs, batch)
                
                running_loss += batch_loss.item()
                count += 1
        
        return running_loss / count

    else:
        model.train()
        for batch in tqdm(dataloader):
            batch = {k: v.to(device) for k, v in batch.items()}

            optimizer.zero_grad()

            outputs = model(**batch)
            batch_loss = compute_loss(model, outputs, batch)

            batch_loss.backward()
            optimizer.step()
            scheduler.step()

            running_loss += batch_loss.item()
            count += 1

        return running_loss / count



for epoch in range(num_epochs):
    train_loss = run_epoch(train_loader, evaluating=False)
    train_losses.append(train_loss)

    val_loss = run_epoch(val_loader, evaluating=True)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
